In [ ]:
pip install pandas

In [ ]:
pip install agentpy

In [ ]:
import pandas as pd

# Model design
import agentpy as ap
import networkx as nx
import random

# Visualization
import numpy as np
import matplotlib.pyplot as plt
import math as math
import seaborn as sns
import IPython

AIDES

https://agentpy.readthedocs.io/en/latest/agentpy_virus_spread.html

https://agentpy.readthedocs.io/en/latest/agentpy_segregation.html

https://docs.python.org/3/library/random.html


In [ ]:
class Person(ap.Agent):

    def setup(self):
        """ Initialize new variables at agent creation. """
        self.grid = self.model.grid
        self.random = self.model.random
        
        self.tps_statut = 0
        self.statut = 0  # Susceptible = 0, Exposed = 1, Infected = 2, Recovered = 3
        
        self.dE = np.random.exponential(3)
        self.dI = np.random.exponential(7)
        self.dR = np.random.exponential(365)
        
        
    def being_sick(self):
        """ Spread disease to peers in the network. """
                  
        if self.statut == 0 :
            self.infected_n = 0
            # neighbors : distance default 1 = nber of cells to cover in each direction, 
            # including diagonally connected cells
            # distance = 1 == 8 neighbors
            for n in self.grid.neighbors(self, distance = 1):
                if n.statut == 2:
                    self.infected_n = self.infected_n + 1
            self.infection_proba = math.exp(-0.5 * self.infected_n)
            if self.random.uniform(0,1) > self.infection_proba:
                self.statut = 1
                self.tps_statut = -1
            else:
                self.tps_statut = self.tps_statut + 1
                    
        
        if self.statut == 1:
            if self.tps_statut >= self.dE:
                self.statut = 2
                self.tps_statut = -1
            else:
                self.tps_statut = self.tps_statut + 1 
        
            
        if self.statut == 2:
            if self.tps_statut >= self.dI:
                self.statut = 3
                self.tps_statut = -1
            else:
                self.tps_statut = self.tps_statut + 1 
                
                
        if self.statut == 3:
            if self.tps_statut >= self.dR:
                self.statut = 0
                self.tps_statut = 0
            else:
                self.tps_statut = self.tps_statut + 1 
                
    def move_rd(self):
        new_spot = self.random.choice(self.model.grid.all)
        self.grid.move_to(self, new_spot)
 

On veut un environnement discret donc on va se servir de 'grid' : n-dimensional spatial topology with discrete positions (vs 'space' poour positions continues)

'network' graph topology consists of AgentNode and edges


grid, space & network contiennent les méthodes suivantes : .add_agents(), .move_to(), .neighbors() ...

In [ ]:
class VirusModel(ap.Model):

    """ Initialize the agents and grid of the model. """
    def setup(self):

        # Create agents and grid        
        self.grid = ap.Grid(self, (self.p.size, self.p.size), 
                            torus = True, check_border = False, track_empty = False)
        print(self.grid)
        print(self.grid.shape)
        
        self.agents = ap.AgentList(self, self.p.population, Person)        
        self.grid.add_agents(self.agents, random = True, empty = False)

        # Infect a random share of the population
        I0 = self.p.infected
        self.agents.random(I0).statut = 2
        
        # Record share of agents with each condition
        for i, c in enumerate(('S', 'E', 'I', 'R')):
            n_agents = len(self.agents.select(self.agents.statut == i))
            self[c] = n_agents / self.p.population
            self.record(c)
            #print(n_agents)
            
    def step(self):
        """ Define the models' events per simulation step. """
        
        # Stop simulation if disease is gone
        self.infected = self.agents.select(self.agents.statut == 2)
        if len(self.infected) == 0:
            self.stop()
            
        self.agents.move_rd()
        #print(self.random.choice(self.model.grid.all))
        #print(self.agents.statut)
        
        # Call 'being_sick' for all agents
        self.agents.being_sick()
    
        # Record share of agents with each condition
        for i, c in enumerate(('S', 'E', 'I', 'R')):
            n_agents = len(self.agents.select(self.agents.statut == i))
            self[c] = n_agents / self.p.population
            self.record(c)
            #print(n_agents)
    
    #def end(self):
        """ Record evaluation measures at the end of the simulation. """
            
        # Record final evaluation measures
        #self.report('Share of Susceptible', self.S)
        #self.report('Share of Exposed', self.E)
        #self.report('Share of Infected', self.I)
        #self.report('Share of Recovered', self.R)

In [ ]:
parameters = {
    'population': 20000,
    'size': 300, # indices de 0 à 299
    'infected': 20,
    
    'steps': 730,
    'seed' : 11290177
}

model = VirusModel(parameters)
results = model.run()

In [ ]:
results

In [ ]:
results.variables.VirusModel

In [ ]:
data = results.variables.VirusModel
ax = data.plot()

In [ ]:
data.to_csv('C:/Users/Hanae/Desktop/VIA IRD/SANTES & TERRITOIRES/2-Donnees/python_h_rd.csv', index=True)

In [ ]:
results.reporters # reporte ce qu'on demande dans def(end)

In [ ]:
class VirusModel_2(ap.Model):

    """ Initialize the agents and grid of the model. """
    def setup(self):

        # Create agents and grid        
        self.grid = ap.Grid(self, (self.p.size, self.p.size), 
                            torus = True, check_border = False, track_empty = False)
        print(self.grid)
        print(self.grid.shape)
        
        self.agents = ap.AgentList(self, self.p.population, Person)        
        self.grid.add_agents(self.agents, random = True, empty = False)

        # Infect a random share of the population
        I0 = self.p.infected
        self.agents.random(I0).statut = 2
        
            
    def step(self):
        """ Define the models' events per simulation step. """
        
        # Stop simulation if disease is gone
        self.infected = self.agents.select(self.agents.statut == 2)
        if len(self.infected) == 0:
            self.stop()
            
        self.agents.move_rd()
        #print(self.random.choice(self.model.grid.all))
        #print(self.agents.statut)
        
        # Call 'being_sick' for all agents
        self.agents.being_sick()
    
    def update(self):
        # Record share of agents with each condition
        for i, c in enumerate(('S', 'E', 'I', 'R')):
            n_agents = len(self.agents.select(self.agents.statut == i))
            self[c] = n_agents / self.p.population
            self.record(c)
            #print(n_agents)
    
    def end(self):
        """ Record evaluation measures at the end of the simulation. """
            
        # Record final evaluation measures
        #self.report('Share of Susceptible', self.S)
        #self.report('Share of Exposed', self.E)
        #self.report('Share of Infected', self.I)
        #self.report('Share of Recovered', self.R)

In [ ]:
parameters = {
    'population': 20000,
    'size': 300, # indices de 0 à 299
    'infected': 20,
    
    'steps': 730,
    'seed' : 11290177
}

model_2 = VirusModel_2(parameters)
results_2 = model_2.run()

In [ ]:
data_2 = results.variables.VirusModel_2
ax = data_2.plot()

In [ ]:
data_2.head()

In [ ]:
data_2.to_csv('C:/Users/Hanae/Desktop/VIA IRD/SANTES & TERRITOIRES/2-Donnees/python_h_rdv.csv', index=True)

# Multi-run experiment

https://agentpy.readthedocs.io/en/latest/overview.html#recording-data

In [ ]:
parameters = {
    'population': 20000,
    'size': 300,
    'infected': 20,

    'steps': 730,
    'seed' : 11290177
}

sample = ap.Sample(
    parameters,
    n = 1
)

In [ ]:
exp = ap.Experiment(VirusModel, sample, iterations = 30, record = True)
results_exp = exp.run()

In [ ]:
results_exp.variables.VirusModel

In [ ]:
results_exp.save()

In [ ]:
results_exp = ap.DataDict.load('VirusModel')

In [ ]:
results_exp.variables.VirusModel.hist();

# Animation plot

In [ ]:
model.grid.attr_grid('statut', otypes = [float])

In [ ]:
model.t

In [ ]:
def animation_plot(model, ax):
    group_grid = model.grid.attr_grid('statut')
    ap.gridplot(group_grid, ax=ax)
    ax.set_title(f"SEIR model \n Time-step: {model.t}, "
                 f"Statut: {model.S}")

fig, ax = plt.subplots()
#model = VirusModel(parameters)
animation = ap.animate(model, fig, ax, animation_plot)

IPython.display.HTML(animation.to_jshtml())

In [ ]:
def animation_plot(m, axs):
    ax1, ax2 = axs
    ax1.set_title("Virus spread")
    ax2.set_title(f"Share infected: {m.I}")

    # Plot stackplot on first axis
    virus_stackplot(m.output.variables.VirusModel, ax1)

    # Plot network on second axis
    color_dict = {0:'w', 1:'b', 2:'r', 3:'g'}
    colors = [color_dict[c] for c in m.agents.statut]

fig, axs = plt.subplots(1, 2, figsize=(8, 4)) # Prepare figure
parameters['population'] = 20000 # Lower population for better visibility
animation = ap.animate(VirusModel(parameters), fig, axs, animation_plot)

In [ ]:
IPython.display.HTML(animation.to_jshtml())

https://agentpy.readthedocs.io/en/latest/_modules/agentpy/grid.html#Grid.neighbors
    
https://agentpy.readthedocs.io/en/latest/reference_grid.html?highlight=neighbors#agentpy.Grid.neighbors